# Navigating 3DV 2026 with Vision-Language Models and Embeddings

**3DV 2026** is the 13th edition of the International Conference on 3D Vision, held in **Vancouver, March 20–23**. 

The conference received a record **404 valid submissions** this year, of which **177 papers were accepted:** 

- 22 for oral presentation
- 155 for poster presentation

For an overall acceptance rate of 43.8%.

The program runs as a single-track conference over 3.5 days: 6 oral sessions (12-minute talks), 6 poster sessions (~1.5 hours each), and 4 keynote talks from.

## The Problem

Even armed with the conference schedule, deciding *which* of 177 papers to prioritise is non-trivial. 

Papers are distributed across 6 poster sessions and 12 oral slots, many running concurrently with talks you also want to see. 

A quick scan of titles is not enough.

IMO, the point of going to a conference is to to understand the research landscape, spot the most distinctive work, and know where the demos and code are.

## The Approach

We load all 177 papers — full PDF page images plus structured metadata — into a [FiftyOne](https://docs.voxel51.com/) grouped dataset, then run three complementary analyses:

#### VLM-based annotation

A reasoning vision-language model reads the first page of each paper and extracts topic labels, author affiliations, and project page links.

#### Document embeddings

Jina v4 encodes every page as a dense vector; per-paper embeddings are computed by mean pooling, with reference pages excluded

#### FiftyOne Brain analysis

UMAP visualizations, similarity indexes, representativeness scores, and uniqueness scores surface the papers most worth your time

#### The result?

The result is an interactive dataset you can explore in the FiftyOne App.

You can filter by topic, sort by uniqueness, drill into any paper's full page sequence, and build your personal conference schedule.

### Downloading the dataset

I've uploaded the dataset (with all the enrichments you will see in this talk) to Hugging Face. You an download it as follows:

First, install FiftyOne:

```bash
pip install fiftyone
```

Then you download the dataset as follows:

```python
import fiftyone as fo
from fiftyone.utils.huggingface import load_from_hub

# Load the dataset
# Note: other available arguments include 'max_samples', etc
dataset = load_from_hub("Voxel51/3dvs2026_papers")

```

In [1]:
import fiftyone as fo

dataset = fo.load_dataset("3dvs2026_papers")

page_1=dataset.select_group_slices(slices="page_01")

## The Dataset

177 Papers as a Grouped FiftyOne Dataset, where each paper's PDF was converted to per-page PNG images. 

The full set is 3,019 images across 176 papers (one submission had no available PDFs) — is loaded into a [FiftyOne grouped dataset](https://docs.voxel51.com/user_guide/groups.html) where:

- Each **group** is one paper, identified by its submission ID

- Each **slice** within a group is one page image: `page_01`, `page_02`, … for main PDF pages and `supp_page_01`, `supp_page_02`, … for supplementary material

#### What is a "Grouped Dataset"?

A grouped dataset in FiftyOne organizes multiple related samples (like different camera views or modalities of the same scene) under a common group identifier, enabling synchronized visualization and multi-modal analysis across different media types. 

Each group acts as a logical unit with named slices (e.g., left/center/right cameras or RGB/mask/point cloud), where each slice contains homogeneous media but groups can span images, videos, and 3D data.

#### Our Groups

The groups in our dataset are unbalanced by design.

The papers with 8 pages have 8 slices, papers with 22 pages have 22. Papers without supplementary material simply have no `supp_page_*` slices. 

FiftyOne handles this natively.

Every sample (page) carries the full paper metadata as fields: `title`, `authors`, `abstract`, `affiliations`, `poster_session`, `oral_session`, `oral_date`, `oral_start_time`, `poster_date`, `poster_start_time`, `pdf_link`, and `supplementary_link`.

#### For the Purposes of this Talk...

Most annotation and analysis work is done on the `page_01` slice.

The cover page containing the title, author list, affiliations, abstract, and teaser figure. 

This is the richest single page for both VLM-based classification and embedding-based similarity.

## Using a Vision-Language Model to Understand Papers at Scale

Reading 177 abstracts is feasible. 

Reading 177 abstracts *and* making considered decisions about which papers to prioritise across 6 poster sessions and 12 oral slots, while also attending talks and having hallway conversations, is not (at least for me).

### What Model Will We Use?

We use [Qwen3.5-9B](https://huggingface.co/Qwen/Qwen3.5-9B), a reasoning vision-language model, applied directly to the first page image of each paper via the [FiftyOne Model Zoo](https://docs.voxel51.com/model_zoo/index.html). 

The first page is the information-dense page: title, authors, affiliations, abstract, and usually a teaser figure. 

Running the model on this single page per paper is both (relatively) efficient and effective.

#### How Will We Use Qwen?

Three annotation passes are run using [`apply_model`](https://docs.voxel51.com/api/fiftyone.core.collections.html#fiftyone.core.collections.SampleCollection.apply_model):

1. **Topic classification** — assign each paper to one of 10 research areas

2. **Affiliation extraction** — extract the list of author institutions

3. **Project page detection** — find links to code, demos, or supplementary sites

#### Using Qwen in FiftyOne

The machine learning landscape moves fast.

New vision models drop weekly, **sometimes daily**, and waiting on a library's release cycle to access them isn't practical. Your own organization likely has **proprietary models it wants to bring into the same workflow** too.

FiftyOne's **[remote source zoo model](https://docs.voxel51.com/model_zoo/remote.html#working-with-remotely-sourced-models)** pattern solves exactly this problem: it lets you *register, download, and apply models* whose definitions live in external GitHub repositories or public URLs, using the *exact same API* you'd use for any built-in zoo model.

This pattern means the community, and your own team, can **ship model integrations on your own timelines**. 

#### The Usage Pattern

- Register the Zoo Model source code

- Download and instantiate the model

- Set any model level generation parameters.

See [this repo](https://github.com/harpreetsahota204/qwen3_5_vl) for more details on using the model.

In [ ]:
import fiftyone as fo
import fiftyone.zoo as foz

# Register and download
foz.register_zoo_model_source(
    "https://github.com/harpreetsahota204/qwen3_5_vl",
    overwrite=True
)

model = foz.load_zoo_model(
    "Qwen/Qwen3.5-9B",
    media_type="image",
)

model.max_new_tokens = 32768

## Step 1 — Topic Classification

With 177 papers spanning a wide range of 3D vision subfields, a flat list of titles is hard to navigate. 

We give the VLM the first page of each paper and ask it to assign exactly one label from a predefined taxonomy of 10 topic areas — from *3D Reconstruction and Novel View Synthesis* through to *Novel Sensors and Specialized Domains*.

#### Put the Model in `classify` mode and Setup the Prompt

We put the model in `classify` operation mode and returns a structured JSON response (`[{"label": "topic_name"}]`), which FiftyOne stores as a [`Classification`](https://docs.voxel51.com/user_guide/using_datasets.html#classification) label. 

This gives us an immediately filterable and groupable taxonomy directly in the [FiftyOne App](https://docs.voxel51.com/user_guide/app.html).

In [ ]:
model.operation = "classify"

topics = [
    "3D Reconstruction and Novel View Synthesis",
    "3D Generation and Editing",
    "Human Avatars and Motion",
    "Hand-Object and Human-Scene Interaction",
    "Semantic 3D Understanding",
    "Autonomous Driving & Outdoor Perception",
    "Depth and Stereo Estimation",
    "Dynamic and 4D Scenes",
    "Geometric Vision Fundamentals",
    "Novel Sensors and Specialized Domains",
]

topics_str = "\n".join(f"- {t}" for t in topics)

prompt = (
    "You are classifying papers accepted to the International Conference on 3D Vision (3DV). "
    "You are given an image of the first page of a paper. "
    "Based on the title, abstract, and any other visible content, "
    "assign the paper to EXACTLY one of the following topics: "
    f"[{topics_str}]. "
    "Choose EXACTLY one of the topic name, exactly as shown above. Respond in this format: "
    '[{"label": "topic_name"}]. '
)

model.prompt = prompt

#### 🏃🏽‍♂️ Now, Run Inference for the Classification Task

In [ ]:
page_1.apply_model(
    model,
    label_field="topic",
    batch_size=16,
    num_workers=4
)

## Step 2 — Extracting Author Affiliations

Knowing which institutions are most active at 3DV, and which papers come from which labs, is useful context for navigating the conference. 

The affiliations appear on `page_01` directly below the author names.


#### Put the Model in `vqa` mode and Setup the Prompt

We prompt the model to return a Python list of institution strings, one entry per affiliation. 

It's just going to be a string, which we will need to parse...

Notice that ASCII normalisation is requested (`ü → u`, `é → e`, `ß → ss`) to avoid encoding inconsistencies when filtering or grouping by institution in the FiftyOne App. 

The parsed list is stored as `affiliations` on the `page_01` samples, and a derived `number_of_authors` integer field is written for quick filtering.

In [ ]:
model.operation="vqa"

model.prompt = (
    "Extract the institution affiliations associated with this academic paper. "
    "This information typically follows the title and list of author names. "
    "Each affiliation should be its own entry."
    "Use standard ASCII characters — replace accented or special characters with their plain ASCII equivalents (e.g. ü → u, é → e, ß → ss). "
    "Don't think too much, its an easy task to do."
    "Report your response as a properly formatted Python list of strings."
)


#### 🏃🏽‍♂️ Now, Run Inference for VQA Mode

In [ ]:
page_1.apply_model(
    model,
    label_field="affiliations",
    batch_size=16,
    num_workers=4
)

## Step 3 — Finding Project Pages and Code

A paper with a live project page or GitHub repository is more accessible than one without.

If the reader is lucky, we might get working code, pre-trained models, or an interactive demo you can visit at the poster. 

For me, this is a useful signal for prioritising **which posters to spend time at**.

The model is given the first page image and asked to report any project page or repository link it finds, delimited by triple backticks. 

If no link is present it says so in prose, which parses to `None`. 

The parsed result is stored as `project_page` (a URL string or `None`) and a derived boolean `has_code` (`"Yes"` / `"No"`) is written for easy filtering in the app.


#### Setup the Prompt and Run Inference

In [ ]:
model.prompt = (
    "Determine if this paper provides a link to their project page, GitHub repository, "
    "or site where additional material can be accessed. "
    "Report this link delimited by triple backticks. "
)

page_1.apply_model(
    model,
    label_field="additional_material",
    batch_size=16,
    num_workers=4
)

## Step 4 — Tagging What Each Paper Actually Contributes

Topic labels tell you *what a paper is about*, but not *what kind of contribution it's making*. 

A paper on novel view synthesis could be a new dataset, a new model, a new method, or a new benchmark, and which one it is changes how you should engage with it at the conference.

A new **dataset** paper is worth a careful poster visit if you work in that area. You may want to use the data yourself.

A new **model** paper is where you go to ask about weights, training recipes, and whether the authors plan to release checkpoints.

A new **method** paper rewards reading the paper more than visiting the poster. The contribution is usually an idea you can port to your own work.

A new **benchmark** paper is worth knowing about even if you don't work on the exact task, because it's likely going to shape what "SOTA" means for the next year or two.

#### How We Tag Contributions

We keep the model in `vqa` mode and ask it to return a JSON object with four booleans: `introduces_dataset`, `introduces_model`, `introduces_method`, and `introduces_benchmark`. 

A single paper can (and often does) contribute in multiple categories. For example, a new method *and* a new benchmark to evaluate it on, so these are not mutually exclusive. 

The parsed booleans get written as `"Yes"`/`"No"` string fields so they're trivially filterable and groupable in the FiftyOne App.

In [ ]:
model.prompt = (
    "You are given an image of the first page of a paper accepted to the International Conference on 3D Vision (3DV). "
    "Based on the title, abstract, and any other content, classify this paper by returning a JSON object "
    "with the following boolean fields: introduces_dataset, introduces_model, introduces_method, has_benchmark. "
    "For example: "
    '{"introduces_dataset": true, "introduces_model": false, "introduces_method": true, "introduces_benchmark": false}. '
    "Respond with only the JSON object, no additional text."
    "Don't think too hard, it should be clear what this paper introduces based on the information you see."
)

page_1.apply_model(
    model,
    label_field="contributions",
    batch_size=16,
    num_workers=4
)

## Parsing Model Responses

Qwen3.5 wraps its chain-of-thought in `<think>...</think>` tags before the final answer—strip those tags during parsing to keep only the structured output.

So we will define utility functions handle the two response formats we require.

### `parse_model_list_response`


This strips the `<think>` block via `rpartition("</think>")`, then uses a greedy regex `\[.*\]` with `re.DOTALL` to find the outermost list literal and evaluates it with `ast.literal_eval`. 

Works whether the model wrapped the list in a markdown code fence, added prose, or returned it bare.


In [ ]:
import ast
import re
import json

def parse_model_list_response(response: str) -> list[str]:
    """Parse a reasoning model response into a list of strings.

    Discards everything inside <think>...</think>, then finds the first
    [...] block in the remaining text and evaluates it as a Python literal.
    Works regardless of whether the model wrapped the list in a code fence,
    added prose around it, or returned it bare. If no </think> token is
    present the entire response is searched, so non-reasoning models are
    handled transparently.
    """
    # rpartition returns ("", "", response) when the token is absent,
    # so after_think is always the text we want to search.
    _, _, after_think = response.rpartition("</think>")

    # Greedy match from the first '[' to the last ']', spanning newlines
    match = re.search(r"\[.*\]", after_think, re.DOTALL)
    if not match:
        raise ValueError(f"No list literal found in response:\n{after_think!r}")

    return ast.literal_eval(match.group())

### `parse_model_url_response`

It's the same `<think>` stripping, then finds content between triple backticks and validates it starts with `http`. 

> 🤦🏽‍♂️ Note: In hindsight I should have also looked for `https`, but it is what it is.

Returns `None` if no valid URL is present, so the field stays `None` rather than storing a garbage string.

In [ ]:
def parse_model_url_response(response: str) -> str | None:
    """Parse a reasoning model response into a URL string, or None.

    Tries three extraction strategies in order:
    1. Content inside triple backticks (strips optional language tag e.g. ```text)
    2. Bare URL anywhere in the text matching http(s)://...
    Returns None when no valid URL is found.
    """
    _, _, after_think = response.rpartition("</think>")
    text = after_think.strip()

    # Strategy 1: triple backtick fence — strip optional language tag on first line
    match = re.search(r"```[^\n`]*\n?(.*?)```", text, re.DOTALL)
    if match:
        candidate = match.group(1).strip()
        if candidate.startswith("http"):
            return candidate

    # Strategy 2: bare URL anywhere in the text
    match = re.search(r"https?://\S+", text)
    if match:
        # Strip trailing punctuation that may not be part of the URL
        url = match.group(0).rstrip(".,;)")
        return url

    return None

#### `parse_model_json_response`

The contributions step returns a JSON object instead of a list or URL, so we need a third parser. It does the same `<think>` stripping as the other two, then finds the outermost `{...}` block and runs it through `json.loads`.

**Two small quirks worth handling:**

1. Qwen sometimes emits Python-style `True`/`False` instead of JSON `true`/`false`. A pair of regex substitutions rewrites them before parsing so `json.loads` doesn't choke.

2. The parsed booleans are coerced to `"Yes"`/`"No"` strings on the way out, matching how `has_code` is stored. Every boolean-ish field in the dataset then filters the same way in the FiftyOne App.

In [ ]:
def parse_model_json_response(response: str) -> dict[str, str]:
    """Parse a reasoning model response into a dict of Yes/No strings.

    Discards everything inside <think>...</think>, then finds the first
    {...} block in the remaining text and parses it as JSON. Boolean
    values are normalized to "Yes"/"No" strings, handling true/True/false/False
    and quoted variants. If no </think> token is present the entire response
    is searched, so non-reasoning models are handled transparently.
    """
    _, _, after_think = response.rpartition("</think>")

    match = re.search(r"\{.*\}", after_think, re.DOTALL)
    if not match:
        raise ValueError(f"No JSON object found in response:\n{after_think!r}")

    # Normalize unquoted Python-style booleans to JSON-compatible lowercase
    raw = match.group()
    raw = re.sub(r"\bTrue\b", "true", raw)
    raw = re.sub(r"\bFalse\b", "false", raw)

    parsed = json.loads(raw)

    return {
        k: "Yes" if v in (True, "true", "True", "yes", "Yes") else "No"
        for k, v in parsed.items()
    }

#### Writing Parsed Values Back to the Dataset

One of the nice things about FiftyOne is how little ceremony is involved in adding a new field to every sample in a dataset.

`dataset.values("affiliations_response")` pulls the raw model outputs out as a list, we run each one through the parser, and then `dataset.set_values("affiliations", ...)` writes the parsed lists back as a brand new `affiliations` field on every sample. 

No schema migration, no loop over samples, no save calls.

In [ ]:
affiliations = [parse_model_list_response(r) for r in dataset.values("affiliations_response")]
dataset.set_values("affiliations", affiliations)

#### We Repeat the Same for the Others

The same two-line pattern is used for every parsed field in the rest of this section: pull the raw responses out with `values`, transform them in plain Python, and set them back with `set_values`.

In [ ]:
url_links = [parse_model_url_response(r) for r in dataset.values("additional_material_response")]
dataset.set_values("project_page", url_links)

n_authors = [len(auth) for auth in dataset.values("authors")]
dataset.set_values("number_of_authors", n_authors)


has_code = ["Yes" if url is not None else "No" for url in dataset.values("project_page")]
dataset.set_values("has_code", has_code)

#### Fanning One Response Out Into Four Fields

Same `values` → parse → `set_values` pattern as before.

The contributions response is a single JSON object with four booleans inside it. 

Rather than store the whole dict as one field, we write each boolean out as its own top-level `"Yes"`/`"No"` field so filtering in the FiftyOne App is a single click per contribution type.

In [ ]:
fields = ["introduces_dataset", "introduces_model", "introduces_method", "introduces_benchmark"]

responses = dataset.values("contributions_response")

parsed = [parse_model_json_response(r) for r in responses]

for field in fields:
    dataset.set_values(field, [p.get(field, "No") for p in parsed])

## Embedding Every Page for Visual Similarity Search

The VLM annotations give us structured labels, but to find papers that are *visually and semantically similar* — regardless of how they were labelled — we need dense vector embeddings.

We use [Jina Embeddings v4](https://huggingface.co/jinaai/jina-embeddings-v4) (`jinaai/jina-embeddings-v4`, retrieval task), a document embedding model that encodes images as 2048-dim vectors optimised for retrieval. 

See the repo [here](https://github.com/harpreetsahota204/jina_embeddings_v4) for details regarding the Remote Source Zoo Model integration.

In [ ]:
import fiftyone as fo
import fiftyone.zoo as foz

# Register this repository as a remote zoo model source
foz.register_zoo_model_source(
    "https://github.com/harpreetsahota204/jina_embeddings_v4",
    overwrite=True 
)

# Load Jina v4 model with desired configuration
emb_model = foz.load_zoo_model(
    "jinaai/jina-embeddings-v4",
    task="retrieval", 
)

#### From Grouped to Flattened

The dataset is a grouped dataset (one group per paper, one slice per page), but for embedding every page we want a flat view where each page is just a sample in its own right.

`select_group_slices(dataset.group_slices)` gives us exactly that: a view containing every slice from every group as a flat collection of 3,019 page samples. 

We save it back to the dataset under the name `flattened_view` so it's available from the App sidebar and any future session without having to rebuild it.

In [ ]:
# Select multiple slices as a flattened view
flattened_view = dataset.select_group_slices(dataset.group_slices)

dataset.save_view("flattened_view", flattened_view)

dataset.save()

### Now, We Can Finally Compute Embeddings

Every page across all 105 slices is embedded — 3,019 vectors in total — using [`compute_embeddings`](https://docs.voxel51.com/api/fiftyone.core.collections.html#fiftyone.core.collections.SampleCollection.compute_embeddings) on the fully flattened view.


It's just a one-liner.

In [ ]:
flattened_view.compute_embeddings(emb_model, embeddings_field=f"jinav4_embeddings")

### Visualizing Embeddings Using FiftyOne Brain

We can use **UMAP visualization** via [`fob.compute_visualization`](https://docs.voxel51.com/api/fiftyone.brain.html#fiftyone.brain.compute_visualization) to project the 2048-dim Jina embeddings down to 2D, giving us an interactive scatter plot of every page in the dataset right inside the FiftyOne App.

When you look at the embeddings panel in the App, you will notice that pages from the same paper naturally cluster together, and reference pages form a visually distinct island that is easy to identify and tag.

In [ ]:
import fiftyone.brain as fob

fob.compute_visualization(
    flattened_view,
    embeddings="jinav4_embeddings",
    brain_key="jina_viz",
    method="umap",
    batch_size=128,
    num_workers=16,
    num_dims=2
)


### Building a Similarity Index 

We can build a **similarity index** using [`fob.compute_similarity`](https://docs.voxel51.com/api/fiftyone.brain.html#fiftyone.brain.compute_similarity), which indexes every page embedding so we can do nearest-neighbour search directly in the App. 

Click any paper page and find the most visually similar pages across the entire conference, no extra code required.

In [ ]:
import fiftyone.brain as fob
# Build a similarity index
fob.compute_similarity(
    flattened_view,
    model="jinaai/jina-embeddings-v4",
    embeddings="jinav4_embeddings",
    brain_key="jina_sim",
    batch_size=128,
    num_workers=16,
)

## Paper-Level Embeddings via Mean Pooling

Per-page embeddings are great for page-level similarity search, but to rank and compare *papers* we want one fixed-size vector per paper, regardless of whether it has 8 pages or 22.

The approach: group page embeddings by `paper_id`, drop any pages tagged `reference-page` (a visually distinct cluster in the UMAP that carries no signal about the contribution), and take the element-wise mean of what's left.

The result is a 2048-dim vector per paper dominated by the method, results, and figure-heavy pages where the actual contribution lives. We write it to every slice of each paper so it's queryable from any view of the dataset.

See [FiftyOne Tags](https://docs.voxel51.com/user_guide/using_datasets.html#tags) for how the `reference-page` tag was applied.

In [ ]:
import numpy as np
from collections import defaultdict

# Pull paper_id, embedding, and tags together
paper_ids, embeddings, tags_list = flattened_view.values(["paper_id", "jinav4_embeddings", "tags"])

# Pool only pages that are NOT tagged as reference-page
groups = defaultdict(list)
for pid, emb, tags in zip(paper_ids, embeddings, tags_list):
    if "reference-page" not in tags:
        groups[pid].append(emb)

# 3. Mean pool each group → one fixed-size vector per paper
pooled = {pid: np.mean(np.stack(embs), axis=0) for pid, embs in groups.items()}

# 4. Map the pooled vector back to every sample in order
pooled_per_sample = [pooled[pid] for pid in paper_ids]

# 5. Write back to the dataset
flattened_view.set_values("all_page_embeddings_pooled", pooled_per_sample)

#### Same Workflow, Paper-Level Embeddings

Same two Brain calls as before (`compute_visualization` for UMAP and `compute_similarity` for nearest-neighbour search), just pointed at `all_page_embeddings_pooled` instead of the per-page embeddings. 

Now the UMAP shows one point per paper and similarity search returns whole papers rather than individual pages.

In [ ]:
import fiftyone.brain as fob

fob.compute_visualization(
    flattened_view,
    embeddings="all_page_embeddings_pooled",
    brain_key="all_page_embeddings_viz",
    method="umap",
    batch_size=128,
    num_workers=16,
    num_dims=2
)

import fiftyone.brain as fob
# Build a similarity index
fob.compute_similarity(
    flattened_view,
    model="jinaai/jina-embeddings-v4",
    embeddings="all_page_embeddings_pooled",
    brain_key="all_page_embeddings_sim",
    batch_size=128,
    num_workers=16,
)

## Ranking Papers — Representativeness and Uniqueness

With per-paper embeddings in place we can score every paper along two complementary axes using the [FiftyOne Brain](https://docs.voxel51.com/brain.html).

1. Representativeness

2. Uniquness

Together these two scores give you a principled shortlist strategy: sort by representativeness to understand the "shape" of the conference; sort by uniqueness to find the outliers most likely to show you something genuinely new.

### Representativeness

We can use [`fob.compute_representativeness`](https://docs.voxel51.com/api/fiftyone.brain.html#fiftyone.brain.compute_representativeness) to 
score each paper by proximity to the nearest cluster centre in embedding space. 

A high score means the paper is an archetypal example of its research area. 

It's the kind of work that defines what a topic is *about* at this edition of the conference.

In [ ]:
import fiftyone.brain as fob

# Compute representativeness scores
fob.compute_representativeness(
    flattened_view,
    representativeness_field="jina_represent",
    method="cluster-center",
    embeddings="all_page_embeddings_pooled"
)

### Uniqueness

We can use [`fob.compute_uniqueness`](https://docs.voxel51.com/api/fiftyone.brain.html#fiftyone.brain.compute_uniqueness) to score each paper by how far it sits from its nearest neighbours. 

A high score means the paper occupies its own region of the embedding space — it is doing something that few or no other papers at 3DV are doing.

In [ ]:
# Detect duplicates using embeddings
results = fob.compute_uniqueness(
    flattened_view,
    uniqueness_field="jina_uniqueness",
    embeddings="all_page_embeddings_pooled"
)


## How to Use What We Just Built?

The dataset is now fully annotated. Here's how to use it to plan your conference days:

**Finding your topic areas**
Open the [FiftyOne App](https://docs.voxel51.com/user_guide/app.html), load the `page_01` view, and filter by `topic` to browse papers in a subfield you care about. The UMAP (`all_page_embeddings_viz`) will show you how papers cluster — papers sitting close together share visual and semantic similarity even across topic labels.

**Surfacing the most distinctive work**
Sort `page_01` by `jina_uniqueness` descending. High-uniqueness papers are the ones doing something genuinely different from the rest of the conference — these are worth a closer look regardless of topic.

**Finding archetypal work in each area**
Sort by `jina_represent` descending within a topic to find the papers that best represent the centre of each research cluster — useful for getting up to speed on a subfield quickly.

**Building your poster shortlist**
Filter by `has_code == "Yes"` for papers with a live project page, then sort by uniqueness. These are likely to have working demos at their poster boards.

**Scheduling around orals**
All oral papers have `oral_date` and `oral_start_time` fields populated. Cross-reference your shortlist against the schedule — oral sessions run for one hour with 12-minute talks, and every oral paper also appears in a poster session the same day.

With 22 orals across 6 oral sessions and 155 posters across 6 poster sessions over 3.5 days, this dataset gives you a systematic way to cut through the noise and make the most of your time in Vancouver.